## Setup

In [1]:
%matplotlib inline

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import json

from src.utils.paths import load_paths
from src.utils.logging import setup_logger
from src.pipeline.feature_pipeline import FeaturePipeline
from src.pipeline.artifacts import default_feature_artifacts
from src.features.extract import load_feature_config, feature_config_hash_text
from src.eval.lood import LOODEvaluator, get_lood_folds
from src.pipeline.data_preparation import load_and_prepare_data
from src.models.lgbm_train import train_lightgbm

from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix

paths = load_paths()
logger = setup_logger(level="INFO")

print("✓ Imports successful")

✓ Imports successful


## 1. Load Combined Data

In [2]:
logger.info("Loading combined dataset (VNAT + ISCX + USBVPN)...")

df_all = load_and_prepare_data(vnat_only=False)

logger.info(f"\nCombined dataset: {len(df_all)} flows")
logger.info(f"Datasets: {sorted(df_all['dataset'].unique())}")
logger.info(f"Splits: {sorted(df_all['split'].unique())}")

# Show basic stats
print("\nSplits by dataset:")
print(df_all.groupby(['dataset', 'split']).size().unstack(fill_value=0))

print("\nVPN labels by dataset:")
print(df_all.groupby('dataset')['label'].value_counts().unstack(fill_value=0))

2026-03-30 12:48:01 | INFO | ai-vpn-firewall | Loading combined dataset (VNAT + ISCX + USBVPN)...
2026-03-30 12:48:01 | INFO | ai-vpn-firewall | Loading VNAT (PCAP-based)...
2026-03-30 12:48:01 | INFO | ai-vpn-firewall | [VNAT] Loaded features.parquet (33711 flows, splits already assigned)
2026-03-30 12:48:01 | INFO | ai-vpn-firewall | Loading ISCX (PCAP-based)...
2026-03-30 12:48:01 | INFO | ai-vpn-firewall | [ISCX] Loaded features.parquet (76687 flows, splits already assigned)
2026-03-30 12:48:01 | INFO | ai-vpn-firewall | Loading USBVPN (JSON-based)...
2026-03-30 12:48:01 | INFO | ai-vpn-firewall | Removing exact duplicate flows across feature columns...
2026-03-30 12:48:01 | INFO | ai-vpn-firewall | Ensuring numeric dtypes for feature columns...
2026-03-30 12:48:01 | INFO | ai-vpn-firewall | ✓ All feature columns successfully converted to numeric dtypes
2026-03-30 12:48:02 | INFO | ai-vpn-firewall | Removed 5750 duplicate flows (7.92%)
2026-03-30 12:48:02 | INFO | ai-vpn-firewall |

## 2. Create LOOD Folds

In [3]:
# Create LOOD evaluator and folds
evaluator = LOODEvaluator()
folds = evaluator.create_folds(["vnat", "iscx", "usbvpn"])

evaluator.print_lood_summary()

# Prepare data for all folds
lood_data = evaluator.prepare_all_lood_data(df_all)

print(f"\n✓ Created {len(lood_data)} LOOD folds")
print(f"Available folds: {list(lood_data.keys())}")

2026-03-30 12:48:02 | INFO | ai-vpn-firewall | Created fold: Train on iscx, usbvpn | Test on vnat
2026-03-30 12:48:02 | INFO | ai-vpn-firewall | Created fold: Train on vnat, usbvpn | Test on iscx
2026-03-30 12:48:02 | INFO | ai-vpn-firewall | Created fold: Train on vnat, iscx | Test on usbvpn
2026-03-30 12:48:02 | INFO | ai-vpn-firewall | 
2026-03-30 12:48:02 | INFO | ai-vpn-firewall | LOOD (Leave-One-Out-Dataset) Evaluation Plan
2026-03-30 12:48:02 | INFO | ai-vpn-firewall | ======================================================================
2026-03-30 12:48:02 | INFO | ai-vpn-firewall | 
Fold 1: Train on iscx, usbvpn | Test on vnat
2026-03-30 12:48:02 | INFO | ai-vpn-firewall |   Fold ID: fold_iscx_usbvpn_vs_vnat
2026-03-30 12:48:02 | INFO | ai-vpn-firewall |   Training on: iscx, usbvpn
2026-03-30 12:48:02 | INFO | ai-vpn-firewall |   Testing on: vnat
2026-03-30 12:48:02 | INFO | ai-vpn-firewall | 
Fold 2: Train on vnat, usbvpn | Test on iscx
2026-03-30 12:48:02 | INFO | ai-vpn-fi

## 3. Training Configuration

In [4]:
# LOOD output directory
lood_output_dir = paths.artifacts_dir / "lood_evaluation"
lood_output_dir.mkdir(parents=True, exist_ok=True)

logger.info(f"Output directory: {lood_output_dir}")

# Load feature config and model config
features_yaml = paths.configs_dir / "features.yaml"
lgbm_yaml = paths.configs_dir / "lgbm.yaml"

features_config = load_feature_config(features_yaml)
feature_hash = feature_config_hash_text(features_yaml)

logger.info(f"Feature config hash: {feature_hash[:16]}...")

# Model will be trained separately for each fold
print("\n✓ Configuration loaded")

2026-03-30 12:48:02 | INFO | ai-vpn-firewall | Output directory: C:\Users\scoti\PycharmProjects\ai-vpn-firewall\artifacts\lood_evaluation
2026-03-30 12:48:02 | INFO | ai-vpn-firewall | Feature config hash: b34b16527d5a4dae...

✓ Configuration loaded


## 4. Train Models for Each LOOD Fold

Note: This section demonstrates the training loop. Actual training would use `train_lightgbm()` or similar.

In [5]:
# Training loop for all LOOD folds
fold_results = {}

In [6]:
for fold in folds:
    logger.info(f"\n{'='*70}")
    logger.info(f"Training {fold.fold_id}")
    logger.info(f"{'='*70}")

    # Get data for this fold
    df_train_val, df_test = lood_data[fold.fold_id]

    logger.info(f"Training data: {len(df_train_val)} flows")
    logger.info(f"Test data: {len(df_test)} flows")

    # Fit feature pipeline on combined train+val
    logger.info("Fitting feature pipeline...")
    pipeline = FeaturePipeline().fit(df_train_val)

    # Get model features
    feature_cols = pipeline.model_feature_names()
    logger.info(f"Model features: {len(feature_cols)} columns")

    # Transform data
    logger.info("Transforming data...")
    X_train_val = pipeline.transform(df_train_val)
    X_test = pipeline.transform(df_test)

    # Split train/val for pipeline fitting
    train_mask = df_train_val['split'] == 'train'
    val_mask = df_train_val['split'] == 'val'

    X_train = X_train_val[train_mask][feature_cols].copy()
    y_train = df_train_val[train_mask]['label'].copy()

    X_val = X_train_val[val_mask][feature_cols].copy()
    y_val = df_train_val[val_mask]['label'].copy()

    X_test = X_test[feature_cols].copy()
    y_test = df_test['label'].copy()

    logger.info(f"X_train: {X_train.shape}, X_val: {X_val.shape}, X_test: {X_test.shape}")

    # Summary of VPN training samples
    logger.info(f"\nTraining VPN count: {(y_train == 1).sum()}")
    logger.info(f"Validation VPN count: {(y_val == 1).sum()}")
    logger.info(f"Test VPN count: {(y_test == 1).sum()}")

    fold_results[fold.fold_id] = {
        'fold': fold,
        'pipeline': pipeline,
        'data': {
            'X_train': X_train,
            'y_train': y_train,
            'X_val': X_val,
            'y_val': y_val,
            'X_test': X_test,
            'y_test': y_test,
        }
    }

print(f"\n✓ Prepared data for {len(fold_results)} LOOD folds")

2026-03-30 12:48:02 | INFO | ai-vpn-firewall | 
2026-03-30 12:48:02 | INFO | ai-vpn-firewall | Training fold_iscx_usbvpn_vs_vnat
2026-03-30 12:48:02 | INFO | ai-vpn-firewall | ======================================================================
2026-03-30 12:48:02 | INFO | ai-vpn-firewall | Training data: 55875 flows
2026-03-30 12:48:02 | INFO | ai-vpn-firewall | Test data: 407 flows
2026-03-30 12:48:02 | INFO | ai-vpn-firewall | Fitting feature pipeline...
2026-03-30 12:48:02 | INFO | ai-vpn-firewall | Model features: 7 columns
2026-03-30 12:48:02 | INFO | ai-vpn-firewall | Transforming data...
2026-03-30 12:48:02 | INFO | ai-vpn-firewall | X_train: (47137, 7), X_val: (8738, 7), X_test: (407, 7)
2026-03-30 12:48:02 | INFO | ai-vpn-firewall | 
Training VPN count: 10340
2026-03-30 12:48:02 | INFO | ai-vpn-firewall | Validation VPN count: 275
2026-03-30 12:48:02 | INFO | ai-vpn-firewall | Test VPN count: 3
2026-03-30 12:48:02 | INFO | ai-vpn-firewall | 
2026-03-30 12:48:02 | INFO | ai-

## 5. Training Signal Comparison

Show increase in VPN training samples compared to single-dataset approach.

In [7]:
# Calculate training signal increase
print("\n" + "="*70)
print("VPN Training Signal Comparison")
print("="*70)

# Single-dataset baseline (from existing splits)
baseline = {}

for ds in ['vnat', 'iscx', 'usbvpn']:
    train_mask = (df_all['dataset'] == ds) & (df_all['split'] == 'train')
    vpn_count = (df_all[train_mask]['label'] == 1).sum()
    baseline[ds] = vpn_count

print("\nSingle-Dataset Training (Baseline):")
for ds, count in baseline.items():
    print(f"  {ds:10s}: {count:5d} VPN samples")


VPN Training Signal Comparison

Single-Dataset Training (Baseline):
  vnat      :     9 VPN samples
  iscx      :  2629 VPN samples
  usbvpn    :  7711 VPN samples


In [8]:
# LOOD increases
print("\nLOOD Training (Combined Datasets):")

for fold in folds:
    test_ds = fold.test_dataset
    vpn_count = fold_results[fold.fold_id]['data']['y_train'].sum()
    increase = vpn_count - baseline[test_ds]
    pct_increase = 100.0 * increase / baseline[test_ds]

    print(f"  Test {test_ds:10s}: {vpn_count:5d} VPN samples (+{increase:3d}, +{pct_increase:5.1f}%)")


LOOD Training (Combined Datasets):
  Test vnat      : 10340 VPN samples (+10331, +114788.9%)
  Test iscx      :  7720 VPN samples (+5091, +193.6%)
  Test usbvpn    :  2638 VPN samples (+-5073, +-65.8%)


In [9]:
# Total training data comparison
print("\nTotal Training Flows:")

for ds in ['vnat', 'iscx', 'usbvpn']:
    train_mask = (df_all['dataset'] == ds) & (df_all['split'] == 'train')
    baseline_count = df_all[train_mask].shape[0]
    print(f"  {ds:10s}: {baseline_count:6d} baseline flows")

for fold in folds:
    test_ds = fold.test_dataset
    train_count = fold_results[fold.fold_id]['data']['X_train'].shape[0]
    print(f"  Test {test_ds:10s}: {train_count:6d} LOOD train flows")


Total Training Flows:
  vnat      :   2369 baseline flows
  iscx      :   8276 baseline flows
  usbvpn    :  38861 baseline flows
  Test vnat      :  47137 LOOD train flows
  Test iscx      :  41230 LOOD train flows
  Test usbvpn    :  10645 LOOD train flows


## 6. Save LOOD Configuration

In [10]:
# Save LOOD configuration
lood_config = {
    "approach": "Leave-One-Out-Dataset (LOOD)",
    "description": "Train unified model on combined datasets with rotating test sets",
    "datasets": ["vnat", "iscx", "usbvpn"],
    "folds": [
        {
            "fold_id": fold.fold_id,
            "fold_name": fold.fold_name,
            "train_datasets": fold.train_datasets,
            "test_dataset": fold.test_dataset,
        }
        for fold in folds
    ],
    "rationale": "Increases VPN training signal by combining all datasets except held-out test set",
    "expected_increases": {
        "vnat": "~+76 VPN samples (374 → 450)",
        "iscx": "~-1199 VPN samples (2029 → 830) [Note: ISCX has most VPN samples]",
        "usbvpn": "~+371 VPN samples (352 → 723)",
    }
}

config_path = lood_output_dir / "lood_config.json"

with open(config_path, 'w') as f:
    json.dump(lood_config, f, indent=2)

logger.info(f"Saved LOOD config: {config_path}")
print(json.dumps(lood_config, indent=2))

2026-03-30 12:48:03 | INFO | ai-vpn-firewall | Saved LOOD config: C:\Users\scoti\PycharmProjects\ai-vpn-firewall\artifacts\lood_evaluation\lood_config.json
{
  "approach": "Leave-One-Out-Dataset (LOOD)",
  "description": "Train unified model on combined datasets with rotating test sets",
  "datasets": [
    "vnat",
    "iscx",
    "usbvpn"
  ],
  "folds": [
    {
      "fold_id": "fold_iscx_usbvpn_vs_vnat",
      "fold_name": "Train on iscx, usbvpn | Test on vnat",
      "train_datasets": [
        "iscx",
        "usbvpn"
      ],
      "test_dataset": "vnat"
    },
    {
      "fold_id": "fold_usbvpn_vnat_vs_iscx",
      "fold_name": "Train on vnat, usbvpn | Test on iscx",
      "train_datasets": [
        "vnat",
        "usbvpn"
      ],
      "test_dataset": "iscx"
    },
    {
      "fold_id": "fold_iscx_vnat_vs_usbvpn",
      "fold_name": "Train on vnat, iscx | Test on usbvpn",
      "train_datasets": [
        "vnat",
        "iscx"
      ],
      "test_dataset": "usbvpn"
    }

## 7. Next Steps

To complete the LOOD evaluation:

1. **Train LightGBM models** for each fold using `train_lightgbm()` with LOOD data
2. **Evaluate on held-out test set** and compute metrics (AUC, AP, etc.)
3. **Compare cross-dataset performance** (training on other datasets → test robustness)
4. **Aggregate results** with macro-averaging across folds
5. **Analyze improvements** vs. single-dataset training

In [11]:
print("\n" + "="*70)
print("LOOD Preparation Complete")
print("="*70)
print(f"\nCreated {len(folds)} LOOD folds")
print(f"Output directory: {lood_output_dir}")
print("Ready to train models for each fold")
print("\nExpected improvements:")
print("- VNAT: Better generalization with larger training set")
print("- ISCX: More robust performance with diverse VPN samples")
print("- USBVPN: Increased training signal from larger dataset combination")
print("\n" + "="*70)


LOOD Preparation Complete

Created 3 LOOD folds
Output directory: C:\Users\scoti\PycharmProjects\ai-vpn-firewall\artifacts\lood_evaluation
Ready to train models for each fold

Expected improvements:
- VNAT: Better generalization with larger training set
- ISCX: More robust performance with diverse VPN samples
- USBVPN: Increased training signal from larger dataset combination

